# Stage 2b — W2V2-L2 Embedding Extraction
### Alertreck · Transfer Learning Pre-processing

This notebook extracts frozen **wav2vec 2.0 layer-2** embeddings from raw audio and saves them as `.npz` shards for use in `04a-train-w2v2-l2.ipynb`.

The encoder (`facebook/wav2vec2-base`) is **fully frozen** and run once. Training the linear classification head never touches the encoder again, so GPU is only needed here.

---

## Pipeline summary

```
Raw audio  →  resample 16 kHz  →  3-second windows (50% hop)
          →  frozen W2V2 encoder  →  hidden_states[2]  →  mean-pool
          →  L2-normalise  →  768-dim embedding  →  .npz shard
```

## Output layout

```
data/processed/w2v2_l2/
  train/          shard_NNN.npz   X: (N, 768) float32   y: (N,) int64
  val/            shard_NNN.npz
  test/           shard_NNN.npz
  train_aug_A/    shard_NNN.npz
  train_aug_B/    shard_NNN.npz
  train_aug_C/    shard_NNN.npz
  manifest.json
```

## Requirements
- GPU accelerator (T4 or P100 recommended)
- Datasets attached: **raw audio** + **alertreck-mel2** (for `splits.json`)

## Cell 1 — Install dependencies

In [1]:
# Install ffmpeg — required to decode MP3 audio files (most dataset classes are MP3)
!apt-get install -q ffmpeg

# torch, torchaudio, librosa, soundfile, tqdm are pre-installed on Kaggle.
# Only install transformers if missing — avoids triggering RAPIDS conflict warnings.
try:
    import transformers
    print(f'transformers {transformers.__version__} already installed')
except ImportError:
    !pip install -q transformers
    import transformers
    print(f'Installed transformers {transformers.__version__}')

import torch, librosa, soundfile, tqdm
print(f'torch      {torch.__version__}')
print(f'librosa    {librosa.__version__}')
print(f'soundfile  {soundfile.__version__}')
print(f'tqdm       {tqdm.__version__}')

# Confirm ffmpeg is on PATH
import subprocess
result = subprocess.run(['ffmpeg', '-version'], capture_output=True, text=True)
print(f'\nffmpeg: {result.stdout.splitlines()[0] if result.returncode == 0 else "NOT FOUND"}')

Reading package lists...
Building dependency tree...
Reading state information...
ffmpeg is already the newest version (7:4.4.2-0ubuntu0.22.04.1).
0 upgraded, 0 newly installed, 0 to remove and 141 not upgraded.
transformers 5.0.0 already installed
torch      2.10.0+cu128
librosa    0.11.0
soundfile  0.13.1
tqdm       4.67.3

ffmpeg: ffmpeg version 4.4.2-0ubuntu0.22.04.1 Copyright (c) 2000-2021 the FFmpeg developers


## Cell 2 — Clone repository

In [2]:
import os
from pathlib import Path

REPO = Path("/kaggle/working/alertreck")

if not REPO.exists():
    !git clone https://github.com/mangaorphy/alertreck.git {REPO}
else:
    print(f"Repo already exists at {REPO}")

print("Repo contents:", list(REPO.iterdir()))

Cloning into '/kaggle/working/alertreck'...
remote: Enumerating objects: 241, done.
remote: Counting objects: 100% (50/50), done.
remote: Compressing objects: 100% (41/41), done.
remote: Total 241 (delta 7), reused 34 (delta 7), pack-reused 191 (from 1)
Receiving objects: 100% (241/241), 41.81 MiB | 26.47 MiB/s, done.
Resolving deltas: 100% (65/65), done.
Repo contents: [PosixPath('/kaggle/working/alertreck/.git'), PosixPath('/kaggle/working/alertreck/models'), PosixPath('/kaggle/working/alertreck/.claude'), PosixPath('/kaggle/working/alertreck/alertrack'), PosixPath('/kaggle/working/alertreck/MY_PORTFOLIO'), PosixPath('/kaggle/working/alertreck/.gitignore'), PosixPath('/kaggle/working/alertreck/docs'), PosixPath('/kaggle/working/alertreck/dashboard'), PosixPath('/kaggle/working/alertreck/scripts'), PosixPath('/kaggle/working/alertreck/notebooks'), PosixPath('/kaggle/working/alertreck/README.md'), PosixPath('/kaggle/working/alertreck/Alertreck_Proposal_Updated.pdf')]


## Cell 3 — Locate datasets and set up paths

Run this cell to find where your Kaggle input datasets are mounted.

In [3]:
# Find splits.json (from the mel dataset)
print("=== Looking for splits.json ===")
!find /kaggle/input -name "splits.json" 2>/dev/null

print("\n=== Kaggle input datasets ===")
!ls /kaggle/input/

=== Looking for splits.json ===
/kaggle/input/datasets/orpheusmanga/alertreck-splits/splits.json

=== Kaggle input datasets ===
datasets


## Cell 4 — Configure paths

Update `MEL_ROOT` and `AUDIO_ROOT` based on what you saw in Cell 3.

In [4]:
import shutil

# ── UPDATE THESE TWO PATHS ──────────────────────────────────────────────────
MEL_ROOT   = Path("/kaggle/input/datasets/orpheusmanga/alertreck-splits")          # folder containing splits.json
AUDIO_ROOT = Path("/kaggle/input/datasets/orpheusmanga/alertreck-dataset") # dataset ROOT (no trailing /dataset)
# ───────────────────────────────────────────────────────────────────────────

# Copy splits.json to the location the script expects
splits_src = MEL_ROOT / "splits.json"
splits_dst = REPO / "data/processed/splits.json"
splits_dst.parent.mkdir(parents=True, exist_ok=True)
shutil.copy(splits_src, splits_dst)
print(f"Copied splits.json → {splits_dst}")

# Symlink REPO/dataset → .../raw-audio-dataset/dataset/
# The script expects REPO/dataset/background_animals/, REPO/dataset/background_wind_rain/, etc.
audio_src    = AUDIO_ROOT / "dataset"
dataset_link = REPO / "dataset"

# Remove a dangling symlink left from a previous run
if dataset_link.is_symlink():
    dataset_link.unlink()
    print("Removed stale symlink")

if not dataset_link.exists():
    dataset_link.symlink_to(audio_src)
    print(f"Symlinked {dataset_link} → {audio_src}")
else:
    print("dataset already exists (not a symlink)")

print("\nsplits.json exists  :", splits_dst.exists())
print("dataset link exists :", dataset_link.exists())
print("dataset target      :", audio_src)

# Quick sanity — list class folders
if dataset_link.exists():
    print("\nClass folders found:")
    for f in sorted(dataset_link.iterdir()):
        if f.is_dir():
            n = len(list(f.iterdir()))
            print(f"  {f.name:<30} {n} files")

Copied splits.json → /kaggle/working/alertreck/data/processed/splits.json
Symlinked /kaggle/working/alertreck/dataset → /kaggle/input/datasets/orpheusmanga/alertreck-dataset/dataset

splits.json exists  : True
dataset link exists : True
dataset target      : /kaggle/input/datasets/orpheusmanga/alertreck-dataset/dataset

Class folders found:
  background_animals             2139 files
  background_wind_rain           3310 files
  threat_chainsaw                782 files
  threat_dog                     1040 files
  threat_gunshot                 3304 files
  threat_human                   1242 files
  threat_vehicle                 1040 files


## Cell 5 — Verify splits.json paths

The file records absolute Kaggle paths from when `audio_preprocessing.py` was first run. If the audio dataset is mounted at a different path now, the cell below will remap them.

In [5]:
import json

splits = json.loads(splits_dst.read_text())

print("=== Sample paths from splits.json ===")
for split_name, items in splits.items():
    cls, path = items[0]
    exists = Path(path).exists()
    print(f"  [{split_name:<6}] {cls:<25} exists={exists}  {path[:80]}")

# Check if any path is broken
broken = [(cls, p) for items in splits.values() for cls, p in items if not Path(p).exists()]
print(f"\nBroken paths: {len(broken)}")
if broken:
    print("  Example broken:", broken[0][1][:80])
    print("\n>>> Update OLD_PREFIX and NEW_PREFIX in the next cell to remap them.")

=== Sample paths from splits.json ===
  [train ] background_animals        exists=False  /Users/cococe/Desktop/alertreck/dataset/background_animals/ds02_bird_peacock__pe
  [val   ] background_animals        exists=False  /Users/cococe/Desktop/alertreck/dataset/background_animals/ds02_bird_parrot__par
  [test  ] background_animals        exists=False  /Users/cococe/Desktop/alertreck/dataset/background_animals/ds02_bird_crow__crow_

Broken paths: 11333
  Example broken: /Users/cococe/Desktop/alertreck/dataset/background_animals/ds02_bird_peacock__pe

>>> Update OLD_PREFIX and NEW_PREFIX in the next cell to remap them.


## Cell 6 — (Optional) Remap broken paths

Only run this cell if Cell 5 reported broken paths.
Set `OLD_PREFIX` to the prefix shown in the broken path, and `NEW_PREFIX` to where the audio dataset is actually mounted.

In [6]:
# ── Run this cell if Cell 5 reported broken paths ──────────────────────────
# splits.json was generated on Mac; paths start with the local Mac prefix.
# Remap them to the Kaggle path where the audio dataset is actually mounted.

OLD_PREFIX = "/Users/cococe/Desktop/alertreck/dataset"
NEW_PREFIX = str(AUDIO_ROOT / "dataset")   # e.g. .../raw-audio-dataset/dataset
# ───────────────────────────────────────────────────────────────────────────

for split_name in splits:
    splits[split_name] = [
        (cls, p.replace(OLD_PREFIX, NEW_PREFIX))
        for cls, p in splits[split_name]
    ]

splits_dst.write_text(json.dumps(splits, indent=2))
print("Paths remapped and splits.json overwritten.")

# Quick sanity check — first 3 entries of each split
for split_name, items in splits.items():
    for cls, p in items[:1]:
        print(f"  [{split_name:<6}] {cls:<25} exists={Path(p).exists()}  {p[:80]}")

Paths remapped and splits.json overwritten.
  [train ] background_animals        exists=True  /kaggle/input/datasets/orpheusmanga/alertreck-dataset/dataset/background_animals
  [val   ] background_animals        exists=True  /kaggle/input/datasets/orpheusmanga/alertreck-dataset/dataset/background_animals
  [test  ] background_animals        exists=True  /kaggle/input/datasets/orpheusmanga/alertreck-dataset/dataset/background_animals


## Cell 7 — Check GPU

In [7]:
import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("VRAM:", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), "GB")

CUDA available: True
GPU: Tesla T4
VRAM: 15.6 GB


## Cell 8 — Run extraction

Generates clean splits + all three curriculum augmentation phases.

**Expected runtime on T4 GPU:**
- Clean splits (train + val + test): ~15 min
- Phase A (1 copy per window): ~20 min
- Phase B (2 copies): ~30 min
- Phase C (3 copies): ~40 min
- **Total: ~1.5–2 hours**

If you get a GPU out-of-memory error, reduce `--batch-size` to 32.

In [8]:
import subprocess, numpy as np
from pathlib import Path

SR = 16_000

# Find one MP3 from each non-human class
audio_root = AUDIO_ROOT / "dataset"
print("=== ffmpeg MP3 preflight check ===\n")

mp3_classes = [d for d in sorted(audio_root.iterdir())
               if d.is_dir() and d.name != "threat_human"]

all_ok = True
for cls_dir in mp3_classes:
    mp3s = list(cls_dir.glob("*.mp3"))
    if not mp3s:
        print(f"  {cls_dir.name:<30}  no MP3 files found — skipping")
        continue
    test_file = mp3s[0]
    cmd = ["ffmpeg", "-i", str(test_file), "-f", "f32le",
           "-ar", str(SR), "-ac", "1", "pipe:1", "-loglevel", "error"]
    result = subprocess.run(cmd, capture_output=True)
    if result.stdout:
        arr = np.frombuffer(result.stdout, dtype=np.float32)
        print(f"  {cls_dir.name:<30}  OK  ({len(arr):,} samples, {len(arr)/SR:.1f}s)")
    else:
        stderr = result.stderr.decode(errors='replace').strip()[:200]
        print(f"  {cls_dir.name:<30}  FAILED")
        print(f"    stderr: {stderr}")
        all_ok = False

print()
if all_ok:
    print("All MP3 classes load successfully — safe to run extraction (Cell 8).")
else:
    print("FAILED: ffmpeg cannot decode MP3 files. Do NOT run Cell 8 yet.")
    print("Fix: make sure Cell 1 ran in THIS session and showed 'ffmpeg: ffmpeg version ...'")

=== ffmpeg MP3 preflight check ===

  background_animals              no MP3 files found — skipping
  background_wind_rain            no MP3 files found — skipping
  threat_chainsaw                 no MP3 files found — skipping
  threat_dog                      no MP3 files found — skipping
  threat_gunshot                  no MP3 files found — skipping
  threat_vehicle                  no MP3 files found — skipping

All MP3 classes load successfully — safe to run extraction (Cell 8).


In [9]:
# Clear any previous (broken) extraction output before re-running.
# This ensures no old single-class shards mix with the new ones.
W2V2_OUT = REPO / "data/processed/w2v2_l2"
if W2V2_OUT.exists():
    import shutil as _shutil
    _shutil.rmtree(W2V2_OUT)
    print(f"Cleared old output: {W2V2_OUT}")
else:
    print("No previous output to clear.")

No previous output to clear.


In [10]:
os.chdir(REPO)

!python3 scripts/prepare_w2v2_embeddings.py \
    --aug-phase A B C \
    --device cuda \
    --batch-size 64

=== Alertreck Stage 2b — W2V2-L2 Embedding Extraction ===
SR         : 16000 Hz  |  clip: 3.0s (48000 samples)  hop: 1.5s (24000 samples)
Encoder    : facebook/wav2vec2-base  →  layer 2 hidden states  →  768-dim L2-normalised
Device     : cuda  |  batch_size: 64
Phases     : ['A', 'B', 'C']
Output     : data/processed/w2v2_l2

File-level splits (from splits.json):
  train    6799 files  background_animals:1283  background_wind_rain:1200  threat_chainsaw:341  threat_dog:624  threat_gunshot:1982  threat_human:745  threat_vehicle:624
  val      2268 files  background_animals:428  background_wind_rain:400  threat_chainsaw:114  threat_dog:208  threat_gunshot:661  threat_human:249  threat_vehicle:208
  test     2266 files  background_animals:428  background_wind_rain:400  threat_chainsaw:113  threat_dog:208  threat_gunshot:661  threat_human:248  threat_vehicle:208

Loading noise pool (16 kHz): 100%|████████████| 100/100 [00:14<00:00,  6.71it/s]
Noise pool : 100 clips

Loading facebook/wav2ve

## Cell 9 — Verify outputs

In [11]:
import numpy as np

W2V2_OUT = REPO / "data/processed/w2v2_l2"

print("=== Output directories ===")
for d in sorted(W2V2_OUT.iterdir()):
    if d.is_dir():
        shards = list(d.glob("*.npz"))
        total  = sum(np.load(s)["X"].shape[0] for s in shards)
        print(f"  {d.name:<18}  {len(shards):>3} shards  {total:>7,} embeddings")

print("\n=== Manifest ===")
manifest = json.loads((W2V2_OUT / "manifest.json").read_text())
for k, v in manifest.items():
    if k not in ("splits", "label_map", "script_sha256"):
        print(f"  {k}: {v}")

print("\n=== Single shard spot-check ===")
shard = np.load(W2V2_OUT / "train/shard_000.npz")
print(f"  X shape : {shard['X'].shape}   dtype: {shard['X'].dtype}")
print(f"  y shape : {shard['y'].shape}   dtype: {shard['y'].dtype}")
print(f"  X range : [{shard['X'].min():.4f}, {shard['X'].max():.4f}]")
print(f"  L2 norm (first 5): {np.linalg.norm(shard['X'][:5], axis=1)}")

=== Output directories ===
  test                  7 shards    6,764 embeddings
  train                16 shards   15,710 embeddings
  train_aug_A          16 shards   15,710 embeddings
  train_aug_B          32 shards   31,420 embeddings
  train_aug_C          48 shards   47,130 embeddings
  val                   7 shards    6,779 embeddings

=== Manifest ===
  seed: 42
  sample_rate: 16000
  clip_seconds: 3.0
  hop_seconds: 1.5
  clip_samples: 48000
  hop_samples: 24000
  embed_dim: 768
  w2v2_model: facebook/wav2vec2-base
  w2v2_layer: 2
  l2_normalised: True
  curriculum_phases: ['A', 'B', 'C']

=== Single shard spot-check ===
  X shape : (1000, 768)   dtype: float32
  y shape : (1000,)   dtype: int64
  X range : [-0.4128, 0.4549]
  L2 norm (first 5): [1.         1.         0.99999994 1.         1.        ]


## Cell 10 — Save as Kaggle dataset

Zip the output folder so you can upload it as a new Kaggle dataset named `alertreck-w2v2-embeddings`.
This dataset will be attached to `04a-train-w2v2-l2.ipynb`.

In [12]:
ZIP_PATH = Path("/kaggle/working/w2v2_embeddings.zip")

print("Zipping outputs — this may take a few minutes...")
!cd {REPO / 'data/processed'} && zip -r {ZIP_PATH} w2v2_l2/

size_mb = ZIP_PATH.stat().st_size / 1e6
print(f"\nZip created: {ZIP_PATH}  ({size_mb:.1f} MB)")
print("\nNext step: go to Data → Upload New Dataset → upload w2v2_embeddings.zip")
print("Name it: alertreck-w2v2-embeddings")

Zipping outputs — this may take a few minutes...
  adding: w2v2_l2/ (stored 0%)
  adding: w2v2_l2/train/ (stored 0%)
  adding: w2v2_l2/train/shard_001.npz (deflated 0%)
  adding: w2v2_l2/train/shard_010.npz (deflated 0%)
  adding: w2v2_l2/train/shard_008.npz (deflated 0%)
  adding: w2v2_l2/train/shard_013.npz (deflated 0%)
  adding: w2v2_l2/train/shard_004.npz (deflated 0%)
  adding: w2v2_l2/train/shard_006.npz (deflated 0%)
  adding: w2v2_l2/train/shard_003.npz (deflated 0%)
  adding: w2v2_l2/train/shard_014.npz (deflated 0%)
  adding: w2v2_l2/train/shard_011.npz (deflated 0%)
  adding: w2v2_l2/train/shard_009.npz (deflated 0%)
  adding: w2v2_l2/train/shard_000.npz (deflated 0%)
  adding: w2v2_l2/train/shard_005.npz (deflated 0%)
  adding: w2v2_l2/train/shard_007.npz (deflated 0%)
  adding: w2v2_l2/train/shard_002.npz (deflated 0%)
  adding: w2v2_l2/train/shard_015.npz (deflated 0%)
  adding: w2v2_l2/train/shard_012.npz (deflated 0%)
  adding: w2v2_l2/manifest.json (deflated 52%)
  ad